# Auditoria de qualidade — standings WNBA

## tl;dr

- Oito runs parciais receberam HTTP 200 e rejeitaram 240 de 240 linhas.
- O payload repetiu `Panevezys Women (784)` nas 30 posições em seis dias UTC distintos.
- O guardrail impediu persistência: zero linhas críticas chegaram ao standings canônico.
- Sete runs perderam o vínculo próprio com raw/issue por deduplicação content-addressed no mesmo dia.

## Context & Methods

Notebook companheiro do relatório `report.html`. A evidência foi consultada em 21/07/2026, em modo somente leitura, nas tabelas `public.hl_ingestion_runs`, `public.hl_ingestion_jobs`, `public.hl_data_quality_issues`, `public.hl_raw_objects` e `public.sports_standings_snapshots`, além do bucket privado `highlightly-raw`.

### Key Assumptions

- O grão é uma execução (`run`) por job.
- Um run é parcial quando rejeita registros ou emite issue crítica.
- As linhas abaixo são o snapshot revisado das consultas live; nenhuma credencial é armazenada.

## Data

### 1. Carregar o snapshot revisado

In [ ]:
partial_runs = [
    {"source_date": date, "http_status": 200, "received": 30, "normalized": 0, "rejected": 30, "linked": date == "2026-07-08"}
    for date in ("2026-07-02", "2026-07-03", "2026-07-04", "2026-07-05", "2026-07-06", "2026-07-07", "2026-07-08", "2026-07-10")
]
critical_history = [
    {"observed_date": date, "rows": 30, "distinct_teams": 1, "team_id": 784, "team_name": "Panevezys Women", "canonical_rows": 0}
    for date in ("2026-07-15", "2026-07-16", "2026-07-17", "2026-07-19", "2026-07-20", "2026-07-21")
]
partial_runs[:3], critical_history[:3]

## Results

### 2. Reconciliar volumes e integridade

In [ ]:
summary = {
    "partial_runs": len(partial_runs),
    "records_received": sum(row["received"] for row in partial_runs),
    "records_normalized": sum(row["normalized"] for row in partial_runs),
    "records_rejected": sum(row["rejected"] for row in partial_runs),
    "runs_with_raw_link": sum(row["linked"] for row in partial_runs),
    "occurrence_days": len(critical_history),
    "canonical_rows": sum(row["canonical_rows"] for row in critical_history),
}
assert summary == {
    "partial_runs": 8, "records_received": 240, "records_normalized": 0,
    "records_rejected": 240, "runs_with_raw_link": 1,
    "occurrence_days": 6, "canonical_rows": 0,
}
summary

### 3. Validar o padrão semântico

In [ ]:
assert all(row["http_status"] == 200 for row in partial_runs)
assert all(row["received"] == row["rejected"] == 30 for row in partial_runs)
assert all(row["distinct_teams"] == 1 for row in critical_history)
assert {(row["team_id"], row["team_name"]) for row in critical_history} == {(784, "Panevezys Women")}
{"rejection_rate": summary["records_rejected"] / summary["records_received"], "canonical_contamination": summary["canonical_rows"]}

## Takeaways

1. Manter o bloqueio dos standings WNBA para interface e modelos.
2. Preservar uma ocorrência bruta por run e incluir `run_id` na identidade da issue.
3. Não repetir standings por data histórica quando o endpoint não aceita data.
4. Usar partidas finalizadas ou fonte alternativa para derivar campanha enquanto a Highlightly não corrigir o endpoint.